# Notebook 04: Results & Interactive Web Map

**Obstacle-Aware Clustering for Geographic Data**

This notebook loads the clustering results from Notebook 03 and publishes them as an interactive web map on ArcGIS Online. The web map provides an accessible, shareable visualization of the obstacle-aware clustering results alongside the standard k-Means baseline.

**Workflow**:
1. Load saved clustering results (no re-computation needed)
2. Prepare the data for ArcGIS (add descriptive labels, select display columns)
3. Authenticate with ArcGIS Online and publish as a hosted feature layer
4. Style and configure the web map in the ArcGIS Online Map Viewer

---

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import json
import getpass

from arcgis.gis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor

## 2. Loading Results from Notebook 03

We load the clustered fire data and optimal parameters saved at the end of Notebook 03. This notebook requires no database access, no boundary projection, and no optimization — all the heavy computation is already done.

In [ ]:
# Load clustered fire data
fires = pd.read_csv('../data/processed/tahoe_fires_clustered.csv')
print(f'Loaded {len(fires)} clustered fires')

# Load optimal parameters
with open('../data/processed/optimal_params.json', 'r') as f:
    opt_params = json.load(f)

print(f'\nTier 1 optimal weights: alpha={opt_params["tier1"]["alpha"]}, '
      f'beta={opt_params["tier1"]["beta"]:.4f}, gamma={opt_params["tier1"]["gamma"]}')
print(f'Tier 2 optimal weights: alpha={opt_params["tier2"]["alpha"]}, '
      f'beta={opt_params["tier2"]["beta"]:.4f}, gamma={opt_params["tier2"]["gamma"]:.4f}')
print(f'\nCluster columns available:')
for col in fires.columns:
    if 'cluster' in col:
        print(f'  {col}')

## 3. Preparing Data for ArcGIS

We add human-readable labels for the cluster assignments and cause types, and select the columns we want to display in the web map popups. ArcGIS will use the LATITUDE and LONGITUDE columns to place the points on the map.

In [ ]:
# Add descriptive cluster labels for the web map
# Tier 2 optimized clusters (primary display)
cluster_labels_t2 = {0: 'Zone 1', 1: 'Zone 2', 2: 'Zone 3'}
fires['Zone_Optimized'] = fires['cluster_t2_opt'].map(cluster_labels_t2)

# Standard k-Means clusters (comparison layer)
# Note: labels_standard was computed on (x,y) only in Notebook 03
# We need to regenerate these since they weren't saved with the same column name
# The standard k-Means labels are in the original notebook output
# We'll use the Tier 2 standard k-Means labels as the comparison
cluster_labels_std = {0: 'Zone 1', 1: 'Zone 2', 2: 'Zone 3'}
fires['Zone_Standard'] = fires['cluster_t2_std'].map(cluster_labels_std)

# Add readable cause label
fires['Cause'] = fires['cause_binary'].map({0: 'Natural (Lightning)', 1: 'Human-Caused'})

# Round numeric columns for cleaner popup display
fires['Fire_Size_Acres'] = fires['FIRE_SIZE'].round(2)
fires['Arc_Length_s'] = fires['s_param'].round(4)

# Select columns for the web map
web_map_cols = [
    'LATITUDE', 'LONGITUDE',
    'Fire_Size_Acres', 'Cause', 'NWCG_GENERAL_CAUSE',
    'FIRE_YEAR',
    'Zone_Optimized', 'Zone_Standard',
    'Arc_Length_s'
]

fires_web = fires[web_map_cols].copy()

print(f'Web map data: {len(fires_web)} fires, {len(web_map_cols)} columns')
print(f'\nColumns:')
for col in web_map_cols:
    print(f'  {col}: {fires_web[col].dtype}')
print(f'\nZone_Optimized distribution:')
print(fires_web['Zone_Optimized'].value_counts().to_string())
print(f'\nZone_Standard distribution:')
print(fires_web['Zone_Standard'].value_counts().to_string())

## 4. Preparing the Lake Tahoe Boundary

We also load the lake boundary coordinates so we can publish them as a separate layer on the web map.

In [ ]:
# Load boundary
boundary_df = pd.read_csv('../data/boundaries/lake_tahoe_boundary.csv')

# Create a boundary CSV with lat/lon for ArcGIS
boundary_df.to_csv('../data/processed/lake_tahoe_boundary_webmap.csv', index=False)
print(f'Boundary: {len(boundary_df)} vertices')

# Save the web map fire data as CSV
fires_web.to_csv('../data/processed/tahoe_fires_webmap.csv', index=False)
print(f'Saved web map data to data/processed/tahoe_fires_webmap.csv')

## 5. Connecting to ArcGIS Online

We authenticate with ArcGIS Online using your credentials. The `getpass` module prompts for your password without displaying it in the notebook output.

**Security note**: Your password is never stored in the notebook — `getpass` handles it securely in memory only.

In [ ]:
# Authenticate with ArcGIS Online
gis = GIS(
    url="https://michele-75.maps.arcgis.com",
    username="Michele-75",
    password=getpass.getpass("Enter your ArcGIS Online password: ")
)

print(f'Connected as: {gis.users.me.username}')
print(f'Organization: {gis.properties.name}')

## 6. Publishing the Fire Data as a Hosted Feature Layer

We upload the prepared CSV and publish it as a hosted feature layer. ArcGIS Online automatically geocodes the points using the LATITUDE and LONGITUDE columns.

In [ ]:
# Upload and publish the fire data
fire_item_properties = {
    'title': 'Lake Tahoe Wildfires - Obstacle-Aware Clustering',
    'description': (
        'Wildfire occurrence data (1992-2020) around Lake Tahoe, clustered using '
        'an obstacle-aware k-Means algorithm. Includes both standard k-Means '
        'and optimized obstacle-aware cluster assignments. '
        'Data source: FPA FOD 6th Edition (USFS Research Data Archive).'
    ),
    'tags': 'wildfires, clustering, Lake Tahoe, obstacle-aware, k-means, portfolio',
    'type': 'CSV',
}

csv_path = '../data/processed/tahoe_fires_webmap.csv'

print('Uploading fire data to ArcGIS Online...')
fire_csv_item = gis.content.add(fire_item_properties, data=csv_path)
print(f'CSV uploaded: {fire_csv_item.title} (ID: {fire_csv_item.id})')

# Publish as a hosted feature layer
print('Publishing as hosted feature layer...')
fire_layer_item = fire_csv_item.publish()
print(f'Published: {fire_layer_item.title}')
print(f'URL: {fire_layer_item.url}')

## 7. Sharing the Layer Publicly

We share the feature layer so it's accessible to anyone with the link — including potential employers viewing your portfolio.

In [ ]:
# Share publicly
fire_layer_item.share(everyone=True)
print(f'Layer shared publicly')
print(f'\nView the layer at:')
print(f'  {fire_layer_item.homepage}')

## 8. Summary of Published Content

The hosted feature layer is now live on ArcGIS Online. The next step is to style it in the **ArcGIS Online Map Viewer** to create the final interactive web map.

In [ ]:
print('PUBLISHED CONTENT SUMMARY')
print('=' * 60)
print(f'\nFeature Layer: {fire_layer_item.title}')
print(f'Item ID:       {fire_layer_item.id}')
print(f'Layer URL:     {fire_layer_item.url}')
print(f'Homepage:      {fire_layer_item.homepage}')
print(f'\nFires:         {len(fires_web)}')
print(f'Fields:        {", ".join(web_map_cols)}')

## 9. Styling the Web Map in ArcGIS Online

With the feature layer published, open the **ArcGIS Online Map Viewer** to create the final web map. Here are the step-by-step instructions:

### Step 1: Open the Map Viewer
- Go to [https://michele-75.maps.arcgis.com](https://michele-75.maps.arcgis.com)
- Click **Map** in the top navigation bar to open a new map
- Click **Add** → **Browse Layers** → **My Content**
- Find "Lake Tahoe Wildfires - Obstacle-Aware Clustering" and click **Add to Map**

### Step 2: Style the Optimized Clusters (Primary Layer)
- In the Layers panel on the left, click the fire layer
- Click **Styles** (paint palette icon)
- Under "Choose an attribute to show", select **Zone_Optimized**
- Choose **Types (Unique Symbols)** as the drawing style
- Click **Style options** to customize colors:
  - **Zone 1** (west/north, mostly human-caused): Orange or red
  - **Zone 2** (south/southeast, human-caused): Yellow or gold  
  - **Zone 3** (east, natural/lightning): Blue or teal
- Adjust symbol size to ~6-8 pixels and set transparency to ~30%
- Click **Done**

### Step 3: Configure Popups
- Click the fire layer → **Pop-ups** (speech bubble icon)
- Click **Title** and set it to: `{Cause} Fire ({FIRE_YEAR})`
- Add fields to display:
  - Fire Size: `{Fire_Size_Acres} acres`
  - Cause: `{NWCG_GENERAL_CAUSE}`
  - Management Zone: `{Zone_Optimized}`
  - Arc-Length Position: `{Arc_Length_s}`
- Click **Done**

### Step 4: Add the Comparison Layer
- The same feature layer contains the `Zone_Standard` field
- Duplicate the layer (right-click → **Duplicate**)
- On the duplicate, change the style attribute to **Zone_Standard**
- Rename this layer to "Standard k-Means" in the Layers panel
- Rename the original layer to "Obstacle-Aware Clustering"
- Turn off the Standard k-Means layer by default (click the eye icon)
- Users can toggle it on to compare the two approaches

### Step 5: Add the Lake Tahoe Basemap Context
- The lake is visible on the default basemap, but you can enhance it:
  - Change the basemap to **Topographic** or **Light Gray Canvas** for a cleaner look
  - Zoom to Lake Tahoe (approximately 39.1°N, -120.05°W, zoom level 11)

### Step 6: Save and Share
- Click **Save** → give the map a title like "Wildfire Clustering Around Lake Tahoe"
- Add a description: "Obstacle-aware k-Means clustering of 1,465 wildfires (1992-2020). Compare standard k-Means vs. obstacle-aware assignments using the layer toggle."
- Click **Share** → **Everyone (public)**
- Copy the shareable URL for your portfolio

### Step 7: Record the Web Map URL
- Paste the shareable URL in the cell below for reference

In [ ]:
# Paste your web map URL here after creating it in the Map Viewer
web_map_url = "PASTE_YOUR_WEB_MAP_URL_HERE"

print(f'Web Map URL: {web_map_url}')
print(f'\nAdd this URL to your GitHub README and portfolio.')

## 10. Discussion

### What This Notebook Demonstrates

This notebook bridges the gap between Python-based data science and professional GIS delivery:

1. **Programmatic data publishing**: The clustering results from Notebook 03's Python analysis pipeline were pushed directly to ArcGIS Online as a hosted feature layer using the `arcgis` Python API. This is a real-world workflow — automating the transfer of analytical results into an enterprise GIS platform.

2. **Interactive web map**: The ArcGIS Online Map Viewer provides the cartographic presentation layer, with styled cluster symbols, informative popups, and a layer toggle for comparing standard vs. obstacle-aware clustering. This is the kind of deliverable that decision-makers (fire managers, land use planners) can explore without any technical background.

3. **Reproducibility**: The full analysis pipeline — from raw FPA FOD data through boundary extraction, projection, clustering, optimization, and publishing — is documented across Notebooks 01-04. Anyone with access to the data can reproduce the results.

### Project Summary

| Notebook | Purpose | Key Output |
|----------|---------|------------|
| 01 - Synthetic Ellipse | Prove the concept with controlled data | Arc-length parameter corrects misassignment |
| 02 - Lake Tahoe Boundary | Extract and parameterize real obstacle | SplineBoundary + boundary CSV |
| 03 - Wildfire Clustering | Full analysis with real fire data | Tier 1 & Tier 2 comparisons, optimal weights |
| 04 - Results & Web Map | Publish and present results | ArcGIS Online interactive web map |